# Tasneem – TinyLlama-1.1B-Chat Evaluation
Testing TinyLlama on medical question answering (open + MCQ).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
SAVE_DIR     = "/content/drive/MyDrive/Colab Notebooks/Activity_1_UFS_1_2026/Tasneem"
JSONL_FILE   = "/content/drive/MyDrive/Colab Notebooks/Activity_1_UFS_1_2026/questions_w_answers.jsonl"
MCQ_INPUT    = "/content/drive/MyDrive/Colab Notebooks/Activity_1_UFS_1_2026/Moaath_Questions_163_189_without_Correct_Answer.csv"
MCQ_GOLD     = "/content/drive/MyDrive/Colab Notebooks/Activity_1_UFS_1_2026/Moaath_Questions_163_189_with_Correct_Answer.csv"

TAG          = "tinyllama"

In [ ]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
import json, os
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# ── Read open questions from jsonl ──────────────────────────────────────
open_bank = []
with open(JSONL_FILE, "r") as f:
    for n, ln in enumerate(f):
        if 116 <= n <= 133:
            d = json.loads(ln)
            open_bank.append({"id": n+1, "q": d["Question"], "keys": d["Must_have"]})

print("open questions:", len(open_bank))

In [ ]:
df_mcq_input = pd.read_csv(MCQ_INPUT)
df_mcq_gold  = pd.read_csv(MCQ_GOLD)
print("MCQ count:", len(df_mcq_input))

In [ ]:
def get_letter(txt):
    txt = txt.upper()
    for c in "ABCDEF":
        if any(p in txt for p in [f"{c})", f"{c}.", f"ANSWER: {c}", f" {c} ", f"({c})"]):
            return c
    return None


def keyword_coverage(text, kw_list):
    if not kw_list:
        return 0.0
    tlow  = text.lower()
    count = sum(
        1 for kw in kw_list
        if sum(w in tlow for w in kw.lower().split()) >= max(1, len(kw.split())//2)
    )
    return count / len(kw_list)

In [ ]:
# ── Load TinyLlama ───────────────────────────────────────────────────────
MOD = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

qcfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MOD)
model = AutoModelForCausalLM.from_pretrained(MOD, quantization_config=qcfg, device_map="auto")

print("TinyLlama ready")

In [ ]:
# ── Open-ended answers ───────────────────────────────────────────────────
open_rows = []

for item in open_bank:
    # TinyLlama uses chatml format
    prompt = (
        "<|system|>\nYou are a helpful medical assistant.</s>\n"
        f"<|user|>\n{item['q']}</s>\n"
        "<|assistant|>\n"
    )
    tok_in = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        tok_out = model.generate(**tok_in, max_new_tokens=200,
                                  temperature=0.3, do_sample=True)

    resp  = tokenizer.decode(tok_out[0][tok_in.input_ids.shape[-1]:], skip_special_tokens=True)
    score = keyword_coverage(resp, item["keys"])

    open_rows.append({"id": item["id"], "question": item["q"],
                      "answer": resp, "must_have_score": score})

open_df = pd.DataFrame(open_rows)
open_df.to_csv(f"{SAVE_DIR}/tinyllama_open_tasneem.csv", index=False)
print("Open mean score:", round(open_df.must_have_score.mean(), 4))

In [ ]:
# ── MCQ answers ──────────────────────────────────────────────────────────
mcq_rows = []

for i, row in df_mcq_input.iterrows():
    q_block = (
        f"{row['Question_text']}\n"
        f"A) {row['(A)']}\nB) {row['(B)']}\nC) {row['(C)']}\n"
        f"D) {row['(D)']}\nE) {row['(E)']}\nF) {row['(F)']}\n"
        "\nAnswer with one letter:"
    )
    prompt = (
        "<|system|>\nYou answer medical MCQs with exactly one letter.</s>\n"
        f"<|user|>\n{q_block}</s>\n"
        "<|assistant|>\n"
    )
    tok_in = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        tok_out = model.generate(**tok_in, max_new_tokens=5, do_sample=False)

    raw   = tokenizer.decode(tok_out[0][tok_in.input_ids.shape[-1]:], skip_special_tokens=True).strip().upper()
    pred  = get_letter(raw) or (raw[-1] if raw else None)
    gold  = df_mcq_gold.loc[i, "Correct_answer"].strip().upper()

    mcq_rows.append({"question": row["Question_text"],
                     f"{TAG}_prediction": pred,
                     "correct": gold, "score": pred == gold})

mcq_df = pd.DataFrame(mcq_rows)
mcq_df.to_csv(f"{SAVE_DIR}/tinyllama_mcq_tasneem.csv", index=False)
print("MCQ accuracy:", round(mcq_df.score.mean(), 4))

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────
summary_df = pd.DataFrame([{
    "model":              TAG,
    "mcq_accuracy":       mcq_df.score.mean(),
    "mcq_correct":        int(mcq_df.score.sum()),
    "mcq_total":          len(mcq_df),
    "open_mean":          open_df.must_have_score.mean(),
    "open_median":        open_df.must_have_score.median(),
    "open_std":           open_df.must_have_score.std(),
    "open_min":           open_df.must_have_score.min(),
    "open_max":           open_df.must_have_score.max(),
    "open_good_0.5":      int((open_df.must_have_score >= 0.5).sum()),
    "open_excellent_0.8": int((open_df.must_have_score >= 0.8).sum()),
}])

summary_df.to_csv(f"{SAVE_DIR}/tinyllama_summary.csv", index=False)
print(summary_df.T)

In [ ]:
import gc
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print("Done – GPU cleared")